### PromptTemplate 
* [PromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.prompt.PromptTemplate.html#langchain_core.prompts.prompt.PromptTemplate)
* [ChatPromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html#langchain_core.prompts.chat.ChatPromptTemplate)
* [ChatMessagePromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.chat.ChatMessagePromptTemplate.html#langchain_core.prompts.chat.ChatMessagePromptTemplate)
* [FewShotPromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.few_shot.FewShotPromptTemplate.html#langchain_core.prompts.few_shot.FewShotPromptTemplate)
* PartialPrompt

In [ ]:
# uv add python-dotenv langchain langchain-openai

In [4]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 사용할 제공자의 키만 있으면 됨 — 설정 여부와 앞 2글자만 확인용으로 출력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


#### LLM 설정 (Groq / OpenAI 선택)

- **Groq**와 **OpenAI** 두 가지 모델을 모두 사용할 수 있습니다. (Groq는 OpenAI 호환 API라서 `ChatOpenAI`로 연결)
- 아래 셀의 `PROVIDER` 값(`"groq"` 또는 `"openai"`)만 바꾸면 이후 모든 예제의 모델이 바뀝니다. 값을 바꾼 뒤에는 이 셀부터 다시 실행하세요.
- 사용할 제공자의 키(`GROQ_API_KEY` 또는 `OPENAI_API_KEY`)가 `.env`에 있어야 합니다. OpenAI를 선택하면 사용 요금이 발생합니다.

In [5]:
from langchain_openai import ChatOpenAI

# 사용할 LLM 제공자 선택: "groq" 또는 "openai" — 이 값만 바꾸면 이후 모든 셀의 모델이 바뀜
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-4o-mini",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    return ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
    )

##### 1) PromptTemplate 의 from_template() 함수 사용
* 주로 LLM(텍스트 완성형 모델, ex. Ollama, GPT-3.5)과 함께 사용
* 하나의 문자열 프롬프트를 생성

In [6]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from pprint import pprint

template_text = "{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요."

# PromptTemplate 인스턴스를 생성
prompt_template = PromptTemplate.from_template(template_text)

llm = get_llm(temperature=0.7)  # 제공자/키/모델명은 공통 설정 셀의 get_llm 사용
print("llm 설정 확인 완료", f"{PROVIDER} / {llm.model_name}")

llm 설정 확인 완료 groq / openai/gpt-oss-120b


In [7]:

chain = prompt_template | llm | StrOutputParser()
response = chain.invoke({"model_name":"ChatGPT", "count":3})
pprint(response)

('ChatGPT는 대규모 텍스트 데이터를 이용해 다음에 올 단어를 예측하도록 설계된 트랜스포머(Transformer) 신경망을 '
 '사전학습합니다. 사전학습이 끝난 뒤에는 인간이 만든 질문‑답변 쌍이나 피드백을 활용해 모델이 원하는 방향으로 행동하도록 조정하는 '
 '강화학습(RLHF) 과정을 거칩니다. 이렇게 두 단계의 학습을 통해 모델은 다양한 주제에 대해 자연스럽고 일관된 답변을 생성할 수 있게 '
 '됩니다.')


##### 2) PromptTemplate 결합하기
* 동일한 Prompt 패턴을 사용하지만 여러 개의 질문을 작성해서 LLM을 실행할 수도 있습니다.

In [8]:
template_text = "{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요."

# PromptTemplate 인스턴스를 생성
prompt_template = PromptTemplate.from_template(template_text)

# 문자열 템플릿 결합 (PromptTemplate + PromptTemplate + 문자열)
combined_prompt = (
              prompt_template
              + PromptTemplate.from_template("\n\n 그리고 {model_name} 모델의 장점을 요약 정리해 주세요")
              + "\n\n {model_name} 모델과 비슷한 AI 모델은 어떤 것이 있나요? 모델명은 {language}로 답변해 주세요."
)
combined_prompt.format(model_name="ChatGPT", count=3, language="한국어")
print(combined_prompt)

chain = combined_prompt | llm | StrOutputParser()
response = chain.invoke({"model_name":"Gemini", "count":3, "language":"한국어"})

pprint(response)

input_variables=['count', 'language', 'model_name'] input_types={} partial_variables={} template='{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요.\n\n 그리고 {model_name} 모델의 장점을 요약 정리해 주세요\n\n {model_name} 모델과 비슷한 AI 모델은 어떤 것이 있나요? 모델명은 {language}로 답변해 주세요.'
('**Gemini 모델의 학습 원리 (3문장)**  \n'
 '1. Gemini는 대규모 텍스트와 멀티모달 데이터(이미지·음성 등)를 사용해 **자기지도 학습**을 수행해, 입력 데이터의 일부분을 '
 '마스킹하고 이를 예측하도록 훈련합니다.  \n'
 '2. 이후 **지도 학습** 단계에서 인간이 만든 질문‑답변, 요약, 번역 등 다양한 태스크 데이터를 이용해 모델이 특정 목표를 '
 '달성하도록 미세조정합니다.  \n'
 '3. 마지막으로 **강화학습‑인간 피드백(RLHF)**을 적용해, 모델이 생성한 답변을 인간 평가자가 점수 매기고, 그 점수를 보상 '
 '신호로 사용해 품질과 안전성을 지속적으로 향상시킵니다.  \n'
 '\n'
 '---\n'
 '\n'
 '### Gemini 모델의 주요 장점 요약  \n'
 '- **멀티모달 이해**: 텍스트뿐 아니라 이미지·음성·동영상까지 동시에 처리·생성할 수 있어 다양한 응용 분야에 활용 가능.  \n'
 '- **높은 정확도와 일관성**: 대규모 사전학습과 RLHF 덕분에 복잡한 추론·대화에서도 일관된 고품질 출력을 제공.  \n'
 '- **안전성·편향 제어**: 사전·사후에 여러 단계의 안전성 검증과 편향 완화 기법을 적용해, 위험한 발언이나 부정확한 정보를 '
 '최소화.  \n'
 '\n'
 '---\n'
 '\n'
 '### Gemini와 비슷한 AI 모델 (한국어 명칭)  \n'
 '- **ChatGPT** (오픈AI)  \n'
 '- **

#### PromptTemplate 의 파라미터를 배열 형태로 하여 여러개 사용하는 경우

In [9]:
# List Comprehension 예제
# before (적용하지 않은 경우)
result_list = []
for val in range(10):
    # 2의
    if val % 2 == 0:
        result_list.append(val ** 2)

result_list        

[0, 4, 16, 36, 64]

In [ ]:
# after (적용한 경우)
result_list2 = [val**2 for val in range(10) if val % 2 == 0]
result_list2

In [10]:
template_text = "{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요."

# PromptTemplate 인스턴스를 생성
prompt_template = PromptTemplate.from_template(template_text)

questions = [
    {"model_name": "GPT-4", "count": 3}, #dict
    {"model_name": "Gemini", "count": 4},
    {"model_name": "Claude", "count": 4},
]

# 여러 개의 프롬프트를 미리 생성
# q변수가 dict 타입
formatted_prompts = [prompt_template.format(**q) for q in questions]
print(formatted_prompts)  # 미리 생성된 질문 목록 확인

for prompt in formatted_prompts:
    print(type(prompt), prompt)
    response = llm.invoke(prompt)
    pprint(response.content)
    print("-" * 50)

['GPT-4 모델의 학습 원리를 3 문장으로 한국어로 답변해 주세요.', 'Gemini 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.', 'Claude 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.']
<class 'str'> GPT-4 모델의 학습 원리를 3 문장으로 한국어로 답변해 주세요.
('GPT‑4는 방대한 텍스트 데이터를 이용해 다음에 올 단어를 예측하도록 학습하는 **자기지도 학습** 방식을 사용합니다. 이 과정에서 '
 '모델은 문맥을 파악하고 의미론적·문법적 패턴을 내부 파라미터에 압축해 저장합니다. 학습이 끝난 뒤에는 인간이 만든 프롬프트에 따라 다양한 '
 '작업을 수행하도록 **미세조정**과 **강화학습(예: RLHF)** 단계가 추가됩니다.')
--------------------------------------------------
<class 'str'> Gemini 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.
('Gemini 모델은 대규모 텍스트와 멀티모달 데이터를 수집해 **사전 학습(Pre‑training)** 단계에서 언어와 이미지 간의 '
 '패턴을 스스로 학습합니다.  \n'
 '다음으로 **지도 학습(Supervised Fine‑tuning)**을 통해 인간이 만든 질문‑답변 쌍이나 코드 예시 등 정확한 정답이 '
 '있는 데이터를 이용해 모델의 출력을 구체화합니다.  \n'
 '그 후 **인간 피드백 강화학습(RLHF, Reinforcement Learning from Human Feedback)**을 적용해, '
 '인간 평가자들이 선호하는 응답을 보상 신호로 사용해 모델을 최적화합니다.  \n'
 '마지막으로 **멀티태스크와 지속적 업데이트**를 수행해 새로운 도메인·언어·이미지 정보를 주기적으로 반영함으로써 성능을 향상시킵니다.')
--------------------------------------------------
<class 'str'> Claude 모델의 학습 원리를 

##### 2) ChatPromptTemplate
* Tuple 형태의 system, user, assistant 메시지 지원
* 여러 개의 메시지를 조합하여 LLM에게 전달 가능
* 간결성과 가독성이 높고 단순한 구조

In [11]:
# 2-튜플 형태의 메시지 목록으로 프롬프트 생성 (type, content)

from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    # role, message
    ("system", "This system is an expert in answering questions about {topic}. \
     Please provide clear and detailed explanations."),
    ("user", "{model_name} 모델의 학습 원리를 설명해 주세요.이 AI 모델은 어느 나라에서 개발되었나요?"),
])

messages = chat_prompt.format_messages(topic="AI", model_name="ExaOne")
print(messages)

# 생성한 메시지를 바로 주입하여 호출하기
print(llm.model_name)
response = llm.invoke(messages)

print(type(response))
print(response.content)

[SystemMessage(content='This system is an expert in answering questions about AI.      Please provide clear and detailed explanations.', additional_kwargs={}, response_metadata={}), HumanMessage(content='ExaOne 모델의 학습 원리를 설명해 주세요.이 AI 모델은 어느 나라에서 개발되었나요?', additional_kwargs={}, response_metadata={})]
openai/gpt-oss-120b
<class 'langchain_core.messages.ai.AIMessage'>
### ExaOne 모델의 학습 원리 (일반적인 대형 언어 모델 관점)

아래 내용은 현재 공개된 대형 언어 모델(Large Language Model, LLM)들의 **전형적인 학습 절차**를 바탕으로 한 설명이며, ExaOne이 구체적으로 어떤 구현 세부 사항을 가지고 있는지는 공식 문서나 논문에서 확인할 수 있는 경우에만 정확히 알 수 있습니다. 따라서 여기서는 “ExaOne”이라는 이름을 가진 모델이 일반적인 트랜스포머 기반 LLM이라고 가정하고, 그 학습 흐름을 단계별로 정리해 보았습니다.

---

#### 1. 데이터 수집 및 전처리
| 단계 | 내용 |
|------|------|
| **대규모 텍스트 코퍼스 구축** | 웹 크롤링, 책, 위키피디아, 뉴스, 포럼, 코드 저장소 등 다양한 출처에서 수십억~수조 토큰 규모의 텍스트를 수집합니다. |
| **클린징(Cleaning)** | HTML 태그 제거, 중복 문서 필터링, 저작권 문제가 있는 콘텐츠 제외, 비속어나 민감 정보 마스킹 등을 수행합니다. |
| **토크나이징(Tokenizing)** | BPE(Byte‑Pair Encoding), WordPiece, 혹은 최근에는 **SentencePiece**와 같은 서브워드 토크나이저를 사용해 텍

In [12]:
# 체인을 생성하여 호출하기
chain = chat_prompt | llm | StrOutputParser()

response = chain.invoke({"topic":"AI", "model_name":"deepseek"})
print(type(response))
print(response)

<class 'langchain_core.messages.base.TextAccessor'>
### DeepSeek 모델의 학습 원리  

DeepSeek 시리즈(예: **DeepSeek‑Chat**, **DeepSeek‑Coder**)는 현재 가장 널리 사용되는 **Transformer 기반의 디코더‑전용 언어 모델**을 기반으로 합니다. 아래에서는 주요 학습 단계와 핵심 기술을 단계별로 정리해 보겠습니다.

---

#### 1. 모델 아키텍처  
| 요소 | 설명 |
|------|------|
| **Transformer Decoder** | Self‑attention 메커니즘을 이용해 입력 토큰들의 관계를 동시에 파악합니다. 층(Layer) 수와 hidden size(차원)는 모델 규모에 따라 다르며, DeepSeek‑Chat‑7B는 32개의 디코더 레이어, hidden size 4096, 32개의 attention heads 를 가집니다. |
| **Byte‑Pair Encoding (BPE) / 토크나이저** | 텍스트를 서브워드 단위 토큰으로 분할해 어휘(vocabulary) 크기를 50k~100k 정도로 유지합니다. 이는 다양한 언어와 특수 기호를 효율적으로 표현하게 해 줍니다. |
| **Positional Encoding** | 순서 정보를 전달하기 위해 Rotary Positional Embedding (RoPE) 등을 사용합니다. 이는 긴 문맥에서도 위치 정보를 정확히 보존합니다. |

---

#### 2. 사전 학습 (Pre‑training)  

| 단계 | 목표 | 손실 함수 | 데이터 |
|------|------|-----------|--------|
| **다음 토큰 예측 (Causal Language Modeling)** | 주어진 문맥을 기반으로 다음 토큰을 맞추는 작업 | Cross‑entropy loss | 공개 웹 크롤링 데이터, 위키피디아, 책, 코드 저장소 등 수백 억 토큰 규모 |
| **대규모 혼합 데이터**

#### 3) ChatPromptTemplate
* SystemMessagePromptTemplate와 HumanMessagePromptTemplate 클래스 사용
* 객체 지향적 접근 - Message 객체를 독립적으로 생성 가능
* 여러 조건에 따라 다른 시스템 메시지 선택

```python
if user_is_beginner:
    system_message = SystemMessagePromptTemplate.from_template("초보자를 위한 설명: {topic}")
else:
    system_message = SystemMessagePromptTemplate.from_template("전문가를 위한 상세 분석: {topic}")
```

In [13]:
# ChatMessagePromptTemplate 활용

from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
    ChatMessagePromptTemplate
)
from langchain_openai import ChatOpenAI

# 개별 메시지 템플릿 정의
system_message = SystemMessagePromptTemplate.from_template(
    "You are an AI expert in {topic}. Please provide clear and detailed explanations."
)
user_message = HumanMessagePromptTemplate.from_template(
    "{question}"
)
ai_message = AIMessagePromptTemplate.from_template(
    "This is an example answer about {topic}."
)

# ChatPromptTemplate로 메시지들을 묶기
chat_prompt = ChatPromptTemplate.from_messages([
    system_message,
    user_message,
    ai_message
])

# 메시지 생성
messages = chat_prompt.format_messages(topic="건강", question="독감예방 접종을 꼭 맞아야 하나요?")

# LLM 호출 (공통 설정 셀의 llm 사용)
response = llm.invoke(messages)

# 결과 출력
print(response.content)

## 독감 예방 접종, 꼭 맞아야 할까?  

아래 내용은 **대한민국 질병관리청(KCDC)**, **세계보건기구(WHO)**, 그리고 주요 의료 기관들의 최신 권고를 바탕으로 정리한 **‘독감(인플루엔자) 예방 접종’**에 대한 종합적인 안내입니다.  

---

## 1. 독감 예방 접종이란?  

| 구분 | 내용 |
|------|------|
| **목적** | 인플루엔자 바이러스에 대한 면역을 미리 형성해 감염 위험을 낮추고, 감염 시 중증·합병증을 감소시킴 |
| **대상 바이러스** | A형(H1N1, H3N2 등)·B형(빅터리아·야마가타) 2~3가지 균주를 포함한 **4가(Quadrivalent) 백신**이 현재 표준 |
| **접종 시기** | 일반적으로 **10월 초~11월 중**에 맞으며, 12월까지도 충분히 효과가 유지됨(접종 후 2주 정도면 보호 시작) |
| **접종 방법** | 근육주사(대퇴부 또는 삼각근) – 0.5 ml 1회 투여 (1세~64세) <br>고령자·만성질환자·특정 고위험군은 0.5 ml 1회 + 필요 시 **추가 용량(고용량) 백신**도 선택 가능 |

---

## 2. 왜 “꼭” 맞아야 할까? – 주요 근거  

### 2.1. 개인 건강 보호
1. **감염 위험 감소**  
   - 평균적인 백신 효과는 **40‑60%** 정도(연도별 바이러스 변이와 매칭 정도에 따라 차이).  
   - 특히 65세 이상, 어린이, 임산부, 만성질환자(당뇨, 심혈관질환, 천식 등)는 감염 시 중증으로 진행될 확률이 2‑5배 높음.  

2. **중증·합병증 예방**  
   - 독감은 폐렴, 심근염, 뇌염 등 심각한 합병증을 일으킬 수 있음.  
   - 백신을 맞은 사람은 **입원·중환자실 입원 위험이 50% 이상 감소**한다는 연구 결과가 다수 보고됨.  

3. **치료 비용 절감**  
   - 감염 시 항바이러스제(오셀테미비르 등)와 입원 치료가 필요할 수 있는데, 예방 접종을 통해 **연간 평균 3~5만원**(의료

#### ChatMessagePromptTemplate는 여러 종류의 메시지(시스템, 인간, AI)를 조합하여 복잡한 프롬프트를 생성할 때 유용합니다.
* SystemMessagePromptTemplate: 이 템플릿은 AI 모델에게 역할을 부여하거나 전반적인 규칙을 설정하는 시스템 메시지를 만듭니다. 위의 예시에서는 "번역을 도와주는 유용한 도우미"라는 역할을 지정합니다.
* HumanMessagePromptTemplate: 이 템플릿은 사용자의 질문이나 요청을 담는 인간 메시지를 만듭니다. 아래의 예시에서는 번역할 텍스트를 입력받습니다.
* ChatPromptTemplate.from_messages: 이 클래스 메서드는 시스템 메시지, 인간 메시지 등 여러 종류의 MessagePromptTemplate 객체들을 리스트로 받아 하나의 채팅 프롬프트 템플릿으로 통합합니다.
* format_messages: 이 메서드는 정의된 템플릿에 실제 값을 채워 넣어 [SystemMessage, HumanMessage] 형태의 리스트를 반환합니다. 이 리스트는 채팅 모델(Chat Model) 에 바로 전달될 수 있습니다.

In [14]:
# 필요한 라이브러리 임포트
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

# 1. SystemMessagePromptTemplate와 HumanMessagePromptTemplate 생성
# SystemMessagePromptTemplate는 모델의 페르소나 또는 기본 지침을 설정합니다.
system_template = "You are a helpful assistant that translates {input_language} to {output_language}."
system_message_prompt = SystemMessagePromptTemplate.from_template(system_template)

# HumanMessagePromptTemplate는 사용자로부터 받는 입력 프롬프트를 정의합니다.
human_template = "{text_to_translate}"
human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)

# 2. ChatPromptTemplate 생성
# 위에서 만든 두 템플릿을 리스트로 묶어 ChatPromptTemplate을 만듭니다.
chat_prompt_template = ChatPromptTemplate.from_messages([system_message_prompt, human_message_prompt])

# 3. 프롬프트 포맷팅
# chat_prompt_template.format_messages()를 사용하여 최종 메시지 리스트를 생성합니다.
# 이 함수는 딕셔너리 형태의 입력 변수를 받습니다.
formatted_prompt = chat_prompt_template.format_messages(
    input_language="English",
    output_language="Korean",
    text_to_translate="Chaining can mean making multiple LLM calls in a sequence. Language models are often non deterministic and can make errors, so making multiple calls to check previous outputs or to break down larger tasks into bite-sized steps can improve results."
)

# 4. 결과 출력
print(formatted_prompt)

# LLM 호출
response = llm.invoke(formatted_prompt)

# 결과 출력
print(response.content)

[SystemMessage(content='You are a helpful assistant that translates English to Korean.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Chaining can mean making multiple LLM calls in a sequence. Language models are often non deterministic and can make errors, so making multiple calls to check previous outputs or to break down larger tasks into bite-sized steps can improve results.', additional_kwargs={}, response_metadata={})]
체이닝은 여러 LLM 호출을 순차적으로 수행하는 것을 의미할 수 있습니다. 언어 모델은 종종 비결정적이며 오류를 일으킬 수 있기 때문에, 이전 출력물을 확인하거나 큰 작업을 작은 단계로 나누어 여러 번 호출하면 결과를 개선할 수 있습니다.


##### 4) FewShotPromptTemplate
* FewShotPromptTemplate은 모델이 특정 형식을 따르게 하거나, 일관된 응답을 생성하도록 유도할 때 유용합니다.
* 도메인 지식이 필요하거나, AI가 오답을 줄이고 더 신뢰할 만한 답변을 생성하도록 해야 할 때 효과적입니다.

##### 4-1) PromptTemplate을 사용하지 않는 경우

In [15]:
# PromptTemplate을 사용하지 않는 경우

# model
print(llm.model_name)

# chain 실행
result = llm.invoke("태양계의 행성들을 간략히 정리해 주세요.")

print(type(result))
print(result.content)

openai/gpt-oss-120b
<class 'langchain_core.messages.ai.AIMessage'>
**태양계 행성 간략 정리**

| 순서(태양으로부터) | 행성 | 주요 특징 |
|-------------------|------|-----------|
| 1 | **수성 (Mercury)** | - 가장 작은 행성, 지구보다 약 0.38배<br>- 대기가 거의 없어 낮·밤 온도 차이가 극심 (−173 °C ~ +427 °C)<br>- 태양에 가장 가까워 88일에 한 바퀴 공전 |
| 2 | **금성 (Venus)** | - 크기·질량이 지구와 비슷한 “지구의 쌍둥이”<br>- 두꺼운 이산화탄소 대기와 황산 구름으로 온실효과가 강해 평균 온도 465 °C<br>- 자전이 역방향이며 243일(자전) / 225일(공전) |
| 3 | **지구 (Earth)** | - 물이 액체 상태로 존재하는 유일한 알려진 행성<br>- 대기 구성: N₂ 78%, O₂ 21% 등<br>- 하나의 위성(달) 보유 |
| 4 | **화성 (Mars)** | - “붉은 행성”이라 불리며 철산화물 때문에 붉은 색<br>- 얇은 CO₂ 대기, 평균 온도 −63 °C<br>- 극지에 물·이산화탄소 얼음, 과거에 물이 흐른 흔적 발견 |
| 5 | **목성 (Jupiter)** | - 가장 큰 행성 (지구의 318배 질량)<br>- 주로 수소·헬륨으로 구성된 가스 행성<br>- 79개의 위성 중 가장 큰 4위성(갈릴리 위성) – 이오, 유로파, 가니메데, 칼리스토<br>- 대적점(거대한 폭풍) 존재 |
| 6 | **토성 (Saturn)** | - 뚜렷한 고리 시스템(얼음·암석 조각)으로 유명<br>- 목성보다 질량은 작지만 부피는 크다 (지구의 95배)<br>- 83개의 위성(2023년 기준) 보유, 가장 큰 위성은 티탄(대기 있음) |
| 7 | **천왕성 (Uranus)** | - 청록색은 메탄 가스가 빛을 흡수하기 때문<br>- 자전축이 약 98° 기울어져 있어 

##### 4-2) FewShotChatMessagePromptTemplate 사용하는 경우

In [16]:
# FewShotChatMessagePromptTemplate 사용하는 경우
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# 예시는 시스템 프롬프트(초등학생 눈높이)와 같은 톤·같은 형식(제목 + 비유 글머리표 + 한 줄 정리)으로 통일
examples_list = [
    {
        "input": "뉴턴의 운동 법칙을 요약해 주세요.",
        "output": """### 뉴턴의 운동 법칙
- **관성**: 달리던 버스가 갑자기 멈추면 몸이 앞으로 쏠리죠? 움직이던 건 계속 움직이려 해요.
- **가속도**: 같은 힘으로 밀면 가벼운 공이 무거운 공보다 더 빨리 굴러가요.
- **작용-반작용**: 벽을 밀면 벽도 나를 똑같이 밀어요.
 한 줄 정리: 힘은 항상 짝으로 움직여요!"""
    },
    {
        "input": "지구의 대기 구성 요소를 알려주세요.",
        "output": """### 지구 대기의 구성
- **질소 (78%)**: 공기의 대부분이에요. 거의 반응하지 않는 얌전한 기체예요.
- **산소 (21%)**: 우리가 숨 쉴 때 꼭 필요해요.
- **이산화탄소 (0.04%)**: 아주 조금이지만 식물이 자라는 데 필요해요.
 한 줄 정리: 공기의 거의 전부는 질소와 산소예요!"""
    }
]

# 예제 프롬프트 템플릿
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

# FewShotChatMessagePromptTemplate 적용
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples_list,
)

# 최종 프롬프트 구성
final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 초등학생도 쉽게 이해할 수 있도록 쉽게 설명하는 과학 교육자입니다."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)

# 모델 생성 및 체인 구성
chain = final_prompt | llm
print(llm.model_name)
print(chain)

# 테스트 실행
result = chain.invoke({"input": "전기의 발생 원리에 대하여 설명해 줘"})
print(result.content)

openai/gpt-oss-120b
first=ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 초등학생도 쉽게 이해할 수 있도록 쉽게 설명하는 과학 교육자입니다.'), additional_kwargs={}), FewShotChatMessagePromptTemplate(examples=[{'input': '뉴턴의 운동 법칙을 요약해 주세요.', 'output': '### 뉴턴의 운동 법칙\n- **관성**: 달리던 버스가 갑자기 멈추면 몸이 앞으로 쏠리죠? 움직이던 건 계속 움직이려 해요.\n- **가속도**: 같은 힘으로 밀면 가벼운 공이 무거운 공보다 더 빨리 굴러가요.\n- **작용-반작용**: 벽을 밀면 벽도 나를 똑같이 밀어요.\n 한 줄 정리: 힘은 항상 짝으로 움직여요!'}, {'input': '지구의 대기 구성 요소를 알려주세요.', 'output': '### 지구 대기의 구성\n- **질소 (78%)**: 공기의 대부분이에요. 거의 반응하지 않는 얌전한 기체예요.\n- **산소 (21%)**: 우리가 숨 쉴 때 꼭 필요해요.\n- **이산화탄소 (0.04%)**: 아주 조금이지만 식물이 자라는 데 필요해요.\n 한 줄 정리: 공기의 거의 전부는 질소와 산소예요!'}], input_variables=[], input_types={}, partial_variables={}, example_prompt=ChatPromptTemplate(input_variables=['input', 'output'], input_types={}, partial_variables={}, messages=[HumanMessagePromptT

#### 5-1) PartialPrompt 
* 프롬프트를 더 동적으로 활용할 수 있으며, AI 응답을 더 일관성 있게 조정 가능함

In [17]:
from datetime import datetime
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 계절을 결정하는 함수 (남반구/북반구 고려)
def get_current_season(hemisphere="north"):
    month = datetime.now().month

    if hemisphere == "north":  # 북반구 (기본값)
        if 3 <= month <= 5:
            return "봄"
        elif 6 <= month <= 8:
            return "여름"
        elif 9 <= month <= 11:
            return "가을"
        else:
            return "겨울"
    else:  # 남반구 (계절 반대)
        if 3 <= month <= 5:
            return "가을"
        elif 6 <= month <= 8:
            return "겨울"
        elif 9 <= month <= 11:
            return "봄"
        else:
            return "여름"

# 프롬프트 템플릿 정의 (부분 변수 적용)
prompt = PromptTemplate(
    template="{season}에 일어나는 대표적인 지구과학 현상은 {phenomenon}이 맞나요? \
        {season}에 주로 발생하는 지구과학 현상을 3개 알려주세요",
    input_variables=["phenomenon"],  # 사용자 입력 필요
    # 함수 자체를 전달 → format() 호출 시점마다 현재 계절을 계산
    partial_variables={"season": get_current_season}  # 동적으로 계절 값 할당
)


# 특정 계절의 현상 질의
query = prompt.format(phenomenon="태풍 발생")
llm = get_llm(temperature=0.7)  # 제공자/키/모델명은 공통 설정 셀의 get_llm 사용
print(llm.model_name)
result = llm.invoke(query)


# 결과 출력
print(f" 프롬프트: {query}")
print(f" 모델 응답: {result.content}")

openai/gpt-oss-120b
 프롬프트: 가을에 일어나는 대표적인 지구과학 현상은 태풍 발생이 맞나요?         가을에 주로 발생하는 지구과학 현상을 3개 알려주세요
 모델 응답: 가을(9‑11월)에는 기후·대기·지구표면이 계절 전환을 맞이하면서 **특유의 지구과학 현상**이 여러 가지 나타납니다.  
대표적인 현상 중 하나가 **태풍(열대저기압)의 발생·접근**이지만, 가을에만 두드러지는 다른 주요 현상도 몇 가지 있습니다. 아래에 가을에 주로 관찰되는 대표적인 지구과학 현상 3가지를 정리해 드립니다.

| # | 현상 | 핵심 내용 | 왜 가을에 주로 나타나는가 |
|---|------|-----------|---------------------------|
| 1 | **태풍·열대저기압의 발생·접근** | - 서태평양에서 형성된 태풍이 한반도·동아시아에 접근하거나 잔여 저기압으로 남아 비·강풍을 동반한다.<br>- 가을은 해수면 온도가 아직 26 °C 이상 유지돼 열대 저기압이 형성되기 충분하고, 동시에 북반구의 편서풍대가 강화돼 태풍이 북쪽으로 이동하기 쉬운 시기이다. | **해수면 온도**가 여름보다 약간 낮아지면서 열대 저기압이 북쪽으로 이동하기 시작하고, **제트기류**가 강해져 태풍이 한반도·동아시아 쪽으로 유입되는 “전환기”이기 때문이다. |
| 2 | **한랭전선·중위도 저기압(전형적인 ‘가을 전선’)의 활발한 통과** | - 남쪽에서 북쪽으로 이동하는 온난·습윤한 대류성 전선이 한랭전선(차가운 대기)과 만나 급격한 기압·온도 변화를 일으킨다.<br>- 강수(보통 장마와는 다른 형태의 폭우·소나기)와 함께 바람이 강해지고, 기온이 급격히 떨어지는 ‘전선 통과’ 현상이 빈번하다.<br>- 전선에 의해 발생하는 **‘전선성 강수’**와 **‘전선성 바람’**은 가을철 농업·교통에 큰 영향을 미친다. | 가을은 **대륙성 고기압**(시베리아·몽골 고기압)이 점차 강해지면서 차가운 북극·시베리아 대기가 남하하고, 동시에 남쪽에서 남아 있

In [18]:
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 계절을 결정하는 함수 (남반구/북반구 고려)
def get_current_season(hemisphere="north"):
    month = datetime.now().month

    if hemisphere == "north":  # 북반구 (기본값)
        if 3 <= month <= 5:
            return "봄"
        elif 6 <= month <= 8:
            return "여름"
        elif 9 <= month <= 11:
            return "가을"
        else:
            return "겨울"
    else:  # 남반구 (계절 반대)
        if 3 <= month <= 5:
            return "가을"
        elif 6 <= month <= 8:
            return "겨울"
        elif 9 <= month <= 11:
            return "봄"
        else:
            return "여름"

# Step 1: 현재 계절 결정
season_name = get_current_season()
#season_name = get_current_season("south")  # 계절 값 얻기
print(f"현재 계절: {season_name}")

현재 계절: 가을


In [19]:

# Step 2: 해당 계절의 자연 현상 추천
prompt2 = ChatPromptTemplate.from_template(
    "{season}에 주로 발생하는 대표적인 지구과학 현상 3가지를 알려주세요. "
    "각 현상에 대해 간단한 설명을 포함해주세요."
)

print(llm.model_name)
# 체인 2: 자연 현상 추천 (입력: 계절 → 출력: 자연 현상 목록)
chain2 = (
    {"season": lambda x : season_name}  # chain1의 출력을 season 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)

# 실행: 현재 계절에 따른 자연 현상 추천
response = chain2.invoke({})
print(f"\n {season_name}에 발생하는 자연 현상:\n{response}")

openai/gpt-oss-120b

 가을에 발생하는 자연 현상:
### 가을에 주로 나타나는 대표적인 지구과학 현상 3가지  

| 현상 | 간단한 설명 |
|------|-------------|
| **1. 제트기류 남하와 저기압 활동 증가** | 가을이 되면 남반구의 차가운 대륙성 기단이 북쪽으로 이동하면서 제트기류가 남쪽으로 내려갑니다. 이로 인해 중위도 저기압(저기압성 전선)이 활발해져 **바람이 강해지고, 급격한 기온 변화와 잦은 비·눈**가 동반됩니다. 특히 동아시아·북미 등에서는 “가을 저기압”이라 불리는 강한 저기압이 자주 발생합니다. |
| **2. 대기 역전 현상·안개·스모그** | 가을밤은 땅이 빠르게 식어 대기 하층이 차가워지고, 상층은 상대적으로 따뜻한 상태가 됩니다. 이 **역전층**이 형성되면 공기 흐름이 억제돼 **산악·계곡·도시 지역에 안개·스모그**가 자주 나타납니다. 역전은 대기 오염 물질이 하층에 머무르게 하여 가시거리를 감소시키고, 호흡기 건강에 영향을 미칠 수 있습니다. |
| **3. 가을 강수·수분 패턴 변화** | 여름에 비해 태양 복사량이 감소하고 대기 중 수증기 함량이 낮아지면서 **강수량이 전반적으로 감소**합니다. 하지만 남쪽에서 북쪽으로 이동하는 차가운 전선이 만나면 **짧은 기간에 집중 호우**가 발생하기도 합니다. 이때 토양은 건조해져 침수 위험은 낮지만, 급격한 강우가 토양 침식·산사태 위험을 높일 수 있습니다. |

> **요약**: 가을에는 제트기류 남하에 따른 저기압·강풍, 대기 역전으로 인한 안개·스모그, 그리고 강수량·수분 패턴의 급격한 변화가 주요 지구과학 현상으로 나타납니다. 이 현상들은 기후·대기·수문학적 특성이 서로 얽혀 가을 특유의 날씨와 환경을 만들어냅니다.


#### 5-2) PartialPrompt 
* API 호출 데이터, 시간 정보, 사용자 정보 등을 반영할 때 매우 유용함

In [20]:
import requests
from datetime import datetime
from langchain_core.prompts import PromptTemplate

EXCHANGE_API_URL = "https://api.exchangerate-api.com/v4/latest/USD"


def get_exchange_rate() -> str:
    """USD→KRW 환율과 API 기준 일자를 문자열로 반환합니다."""
    response = requests.get(EXCHANGE_API_URL, timeout=10)
    response.raise_for_status()
    data = response.json()
    base_date = data.get("date", datetime.now().strftime("%Y-%m-%d"))
    return f"{base_date} 기준 1달러 = {data['rates']['KRW']}원"


# Partial Prompt 활용 (함수 자체를 전달 → format() 호출 시점마다 실행)
prompt = PromptTemplate(
    template=(
        "오늘 날짜: {today}\n"
        "환율 정보: {info}\n\n"
        "위 데이터만 근거로 환율 상황을 간단히 분석해 주세요. "
        "제공되지 않은 과거 수치나 뉴스는 추측해서 쓰지 마세요."
    ),
    input_variables=[],  # 사용자 입력 없음
    partial_variables={
        "today": lambda: datetime.now().strftime("%Y-%m-%d"),
        "info": get_exchange_rate,  # API에서 가져온 데이터 자동 반영
    },
)

# LLM 모델명
print(llm.model_name)

# 프롬프트를 한 번만 생성해 재사용 (API 중복 호출 방지)
formatted = prompt.format()

# 모델에 프롬프트 전달 및 응답 받기
response = llm.invoke(formatted)

# 결과 출력
print(" 프롬프트:", formatted)
print(" 모델 응답:", response.content)

openai/gpt-oss-120b
 프롬프트: 오늘 날짜: 2026-10-08
환율 정보: 2026-10-08 기준 1달러 = 1339.09원

위 데이터만 근거로 환율 상황을 간단히 분석해 주세요. 제공되지 않은 과거 수치나 뉴스는 추측해서 쓰지 마세요.
 모델 응답: **환율 현황 (2026‑10‑08)**  
- 1 USD = **1,339.09 KRW**

### 간단 분석

| 항목 | 의미 |
|------|------|
| **달러당 원화 금액** | 현재 1 달러를 사기 위해 1,339.09 원(원화)이 필요합니다. |
| **원화 기준 구매력** | 1 백만 원(KRW)으로 약 **747 USD**를 구매할 수 있습니다(1,000,000 ÷ 1,339.09 ≈ 747). |
| **수입·수출·여행 비용** | <ul><li>**수입**: 달러 표시 원자재·제품을 구매할 경우, 1 달러당 1,339 원을 지불해야 하므로 비용이 상대적으로 높아집니다.</li><li>**수출**: 외국 바이어가 원화로 결제할 경우, 동일 금액의 원화가 더 적은 달러로 전환되므로 수출 기업 입장에서는 유리할 수 있습니다.</li><li>**여행**: 미국 여행 시, 1 달러당 1,339 원을 지불해야 하므로 여행 경비가 상승합니다.</li></ul> |
| **환전** | 1 USD를 원화로 환전하면 1,339.09 원을 받게 되며, 반대로 1,339.09 원을 달러로 환전하면 1 USD를 얻게 됩니다. |

### 요약
- 오늘(2026‑10‑08) 기준 **달러당 1,339.09 원**이라는 환율이 적용됩니다.  
- 이 환율 하에서는 원화가 1 달러당 1,339 원을 요구하므로, 달러 표시 비용(수입, 해외 여행 등)은 상대적으로 높아지고, 원화로 표시된 수출 매출은 달러로 환산했을 때 상대적으로 유리할 수 있습니다.  

> **주의**: 과거 환율 변동이나 향후 전망에 대한 분석은 제공된 데이터 외의 정보가 없으므로 진행할 수 없습니다.
